# 1 - Introdução

O controlador Proporcional-Integral-Derivativo (PID) representa a arquitetura de controle em malha fechada mais utilizada no mundo. Desde a sua formalização matemática inicial para o direcionamento automático de embarcações, o PID estabeleceu-se como o padrão de fato da automação industrial devido ao seu equilíbrio ideal entre **eficiência de desempenho, robustez e simplicidade conceitual**. Ele funciona da seguinte maneira:

- Ação Proporcional ($K_p$): Atua no presente, fornecendo uma resposta direta e proporcional ao erro instantâneo $e(t) = r(t) - y(t)$.
  
- Ação Integral ($K_i$): Atua no histórico acumulado do erro, eliminando o erro em regime permanente (*offset*) inerente a perturbações de carga.
  
- Ação Derivativa ($K_d$): Prevê a tendência futura do erro avaliando sua taxa de variação $\frac{de(t)}{dt}$, introduzindo amortecimento e aumentando a estabilidade transitória.

Embora a maioria dos processos industriais reais exiba comportamentos não lineares, atrasos de transporte e acoplamentos dinâmicos, um controlador PID bem sintonizado consegue estabilizar e operar com excelente desempenho a grande maioria das plantas em torno de seus pontos de operação nominais.

Apesar disso, o gargalo no uso do PID reside na escolha ideal dos ganhos ($K_p, K_i, K_d$). É exatamente nessa lacuna que a **Programação Não Linear (PNL)** e os conceitos de **Tradeoff e Regularização** se tornam indispensáveis. Ao formular a sintonia como um problema de otimização contínua restrita

$$\min_{K_p, K_i, K_d} J(K_p, K_i, K_d) = \int_0^T t \cdot |e(t)| \, dt + \lambda \int_0^T u(t)^2 \, dt$$

# 2 - Modelo Matemático

### 2.1 Estrutura da Malha Fechada
Estabelecendo a derivação matemática rigorosa do problema de otimização, explicitando as hipóteses de modelagem, a formalização das variáveis de decisão, a função objetivo ponderada e o conjunto de restrições operacionais.

Considera-se um sistema dinâmico linear e invariante no tempo (LTI) representado pela sua função de transferência em tempo contínuo $G(s) = \frac{Y(s)}{U(s)}$, onde $Y(s)$ é a saída controlada e $U(s)$ é o sinal de atuação.

Para a aplicação prática, adota-se o modelo clássico de posicionamento/velocidade de um Motor DC com carga, cuja dinâmica nominal de segunda ordem é dada por:

$$G(s) = \frac{K}{s(\tau s + 1)} = \frac{Y(s)}{U(s)}$$

em que $K$ é o ganho estático e $\tau$ é a constante de tempo do sistema. No domínio do tempo, a relação diferencial entre a atuação $u(t)$ e a resposta $y(t)$ é descrita pela equação diferencial:

$$\tau \ddot{y}(t) + \dot{y}(t) = K u(t)$$

O controlador PID opera realimentando o erro de rastreamento $e(t) = r(t) - y(t)$, em que $r(t)$ é o sinal de referência (degrau unitário). A lei de controle em tempo contínuo é expressa por:

$$u(t) = K_p e(t) + K_i \int_{0}^{t} e(\tau) \, d\tau + K_d \frac{de(t)}{dt}$$

---

### 2.2 Variáveis de Decisão e Domínio de Busca

O objetivo da otimização é determinar os ganhos ótimos do controlador PID. Assim, define-se o vetor de variáveis de decisão $\boldsymbol{x} \in \mathbb{R}^3$ como:

$$\boldsymbol{x} = \begin{bmatrix} K_p \\ K_i \\ K_d \end{bmatrix}$$

Para garantir a estabilidade do sistema em malha fechada e respeitar os limites operacionais físicos dos circuitos ou do algoritmo de controle, impõem-se limites de caixa  inferiores e superiores sobre as variáveis de decisão:

$$  K_{p,\mathrm{min}} \le K_p \le K_{p,\mathrm{max}}$$
$$ K_{i,\mathrm{min}} \le K_i \le K_{i,\mathrm{max}}, $$
$$ K_{d,\mathrm{min}} \le K_d \le K_{d,\mathrm{max}} $$

---

### 2.3 Formulação da Função Custo Multiobjetivo (Tradeoff e Regularização)
A sintonia de controladores envolve inerentemente dois objetivos conflitantes:

1. **Desempenho Temporal ($J_1$)**: Minimizar o erro acumulado entre a resposta $y(t)$ e a referência $r(t)$.
2. **Esforço de Atuação ($J_2$)**: Minimizar a energia do sinal de controle $u(t)$ para evitar a saturação e o desgaste do atuador.

Para quantificar o **desempenho temporal**, adota-se o critério **ITAE** (*Integral of Time-weighted Absolute Error*), que penaliza erros que persistem ao longo do tempo:

$$J_1(\boldsymbol{x}) =\int_{0}^{T_{sim}} t \cdot |e(t, \boldsymbol{x})| \, dt$$

Para quantificar a **energia do sinal de controle** e introduzir **suavidade na resposta da atuação**, adota-se a **Regularização $L_2$ (Tikhonov / Ridge)**:

$$J_2(\boldsymbol{x}) = \int_{0}^{T_{\mathrm{sim}}} [u(t, \boldsymbol{x})]^2 \, \mathrm{d}t = \Vert{}u(\cdot, \boldsymbol{x})\Vert{}_{\mathcal{L}_2}^2$$

Formula-se o problema multiobjetivo combinando ambos os critérios em uma função custo escalar parametrizada pelo peso de regularização $\lambda > 0$:

$$\min_{\boldsymbol{x} \in \Omega_{\boldsymbol{x}}} J(\boldsymbol{x}; \lambda) = J_1(\boldsymbol{x}) + \lambda J_2(\boldsymbol{x}) = \int_{0}^{T_{sim}} \left( t \cdot \vert{}e(t, \boldsymbol{x})\vert{} + \lambda u(t, \boldsymbol{x})^2 \right) dt$$

A variação contínua do parâmetro de regularização $\lambda \in (0, \infty)$ permite mapear a Fronteira Pareto-ótima (Curva de Pareto) no plano $J_2(\boldsymbol{x}_\lambda^*)$ versus $J_1(\boldsymbol{x}_\lambda^*)$, evidenciando o *trade-off* entre precisão e consumo de energia.

---

### 2.4 Discretização Numérica e Integração Temporal
Como as trajetórias temporais $e(t, \boldsymbol{x})$ e $u(t, \boldsymbol{x})$ são obtidas por simulação numérica em um horizonte $T_{\mathrm{sim}}$ com passo de amostragem $\Delta t$, a integral contínua é aproximada pela regra dos trapézios com $N = \frac{T_{\mathrm{sim}}}{\Delta t}$ pontos de amostragem ($t_k = k \cdot \Delta t$):

$$J_1(\boldsymbol{x}) \approx \sum_{k=0}^{N-1} \frac{t_k \vert{}e(t_k, \boldsymbol{x})\vert{} + t_{k+1} \vert{}e(t_{k+1}, \boldsymbol{x})\vert{}}{2} \Delta t$$

$$J_2(\boldsymbol{x}) \approx \sum_{k=0}^{N-1} \frac{[u(t_k, \boldsymbol{x})]^2 + [u(t_{k+1}, \boldsymbol{x})]^2}{2} \Delta t$$

A avaliação de $J(\boldsymbol{x}; \lambda)$ exige a resolução da equação diferencial de malha fechada a cada iteração do otimizador.


--- 




### 2.5 Formato Padrão de Programação Não Linear (PNL)
Incorporando as restrições físicas de saturação do atuador ($|u(t)| \le U_{\max}$) e o sobressinal máximo aceitável ($M_p \le M_{\max}$), o problema de otimização assume o formato padrão de Programação Não Linear contínua:

$$
\begin{aligned}
\min_{\boldsymbol{x} \in \mathbb{R}^3} \quad & J(\boldsymbol{x}; \lambda) = \int_{0}^{T_{\text{sim}}} \left( t \cdot |e(t, \boldsymbol{x})| + \lambda u(t, \boldsymbol{x})^2 \right) dt \\
\text{sujeito a} \quad & g_1(\boldsymbol{x}) = M_p(\boldsymbol{x}) - M_{\max} \le 0 \quad \text{(Sobressinal máximo)} \\
& g_2(\boldsymbol{x}) = \max_{t} |u(t, \boldsymbol{x})| - U_{\max} \le 0 \quad \text{(Saturação do atuador)} \\
& K_{p,\min} \le K_p \le K_{p,\max} \\
& K_{i,\min} \le K_i \le K_{i,\max} \\
& K_{d,\min} \le K_d \le K_{d,\max}
\end{aligned}
$$

---

### 2.6 Condições Teóricas de Otimalidade (1ª e 2ª Ordem)
Sob o ponto de vista da teoria de Programação Não Linear apresentada na disciplina, a caracterização de uma solução ótima $\boldsymbol{x}^* = [K_p^*, K_i^*, K_d^*]^T$ fundamenta-se nas seguintes condições formais:

1. **Condição Necessária de 1ª Ordem (Ponto Estacionário)**: Se $\boldsymbol{x}^*$ é um minimizador local no interior do domínio viável, o vetor gradiente da função objetivo deve anular-se [6, 9]:

$$
\nabla J(\boldsymbol{x}^*) = \begin{bmatrix}
\frac{\partial J}{\partial K_p} \\[4pt]
\frac{\partial J}{\partial K_i} \\[4pt]
\frac{\partial J}{\partial K_d}
\end{bmatrix} = \boldsymbol{0}
$$

1. **Condição Necessária de 2ª Ordem (Curvatura)**: A matriz Hessiana $\nabla^2 J(\boldsymbol{x}^*)$, que codifica a curvatura local da função custo, deve ser semidefinida positiva:

$$
\boldsymbol{d}^T \nabla^2 J(\boldsymbol{x}^*) \boldsymbol{d} \ge 0, \quad \forall \boldsymbol{d} \in \mathbb{R}^3
$$

3. **Condição Suficiente de 2ª Ordem**: Se $\nabla J(\boldsymbol{x}^*) = \boldsymbol{0}$ e a matriz Hessiana for definida positiva ($\nabla^2 J(\boldsymbol{x}^*) \succ 0$), então $\boldsymbol{x}^*$ é garantidamente um minimizador local estrito.

> **Nota de Não-Convexidade**: Como a resposta em malha fechada $y(t, \boldsymbol{x})$ depende não linearmente dos ganhos $\boldsymbol{x}$, a superfície $J(\boldsymbol{x})$ pode apresentar múltiplos mínimos locais e regiões não convexas [4, 12]. Por essa razão, a análise computacional no notebook utilizará algoritmos com restrições (ex.: SLSQP/Ponto Interior) e a técnica de *Multistart* (reinícios aleatórios) para validar se a solução obtida aproxima-se do minimizador global.


---

# 3 - Solução Computacional

Nesta seção, implementa-se computacionalmente a solução do problema de sintonia ótima do controlador PID formulado na Seção 2\. O pipeline é dividido em blocos modulares: definição do modelo dinâmico, simulação da malha fechada, definição da função custo multiobjetivo, otimização restrita via \`scipy.optimize\`, construção da fronteira de Pareto e análise de múltiplos pontos de partida (\*multistart\*).


---


### 3.1. Importação de Bibliotecas e Definição da Planta 
Adota-se o modelo de um servomecanismo de posicionamento (Motor DC) com função de transferência:

$$
G(s) = \frac{K}{s(\tau s + 1)} = \frac{2}{s(0.5 s + 1)} = \frac{4}{s^2 + 2s}
$$

In [ ]:
import numpy as np
import scipy.signal as signal
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# Configuração de estilo dos gráficos
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4

# Parâmetros físicos da planta (Motor DC)
K_planta = 2.0      # Ganho estático
tau_planta = 0.5    # Constante de tempo (s)

# Função de transferência da planta: G(s) = K / (tau*s^2 + s)
num_G = [K_planta]
den_G = [tau_planta, 1.0, 0.0]
G_planta = signal.TransferFunction(num_G, den_G)

print("Planta G(s):")
print(f"Numerador: {num_G}")
print(f"Denominador: {den_G}")

### 3.2. Simulação Temporal em Malha Fechada

Para evitar singularidades impulsivas na ação derivativa ideal ($K_d s$), implementa-se a ação derivativa filtrada clássica:

$$
C(s) = K_p + \frac{K_i}{s} + \frac{K_d s}{\tau_f s + 1}
$$

com constante de filtro $\tau_f = 0.01\text{ s}$. As funções de transferência de malha fechada para a saída $y(t)$ e o sinal de controle $u(t)$ em resposta a um degrau unitário $r(t) = 1$ são:

$$
T_{yr}(s) = \frac{C(s)G(s)}{1 + C(s)G(s)}, \qquad T_{ur}(s) = \frac{C(s)}{1 + C(s)G(s)}
$$

In [2]:
def simular_malha_fechada(ganhos, t_sim=5.0, dt=0.01):
    """
    Simula a resposta temporal da malha fechada ao degrau unitário.
    Retorna: t, y (saída), e (erro), u (sinal de controle), estavel (bool)
    """
    Kp, Ki, Kd = ganhos
    tau_f = 0.01  # Filtro da derivada (100 rad/s)

    # Numerador e denominador de C(s) com filtro derivativo:
    # C(s) = ( (Kp*tau_f + Kd)*s^2 + (Kp + Ki*tau_f)*s + Ki ) / ( tau_f*s^2 + s )
    num_C = [Kp * tau_f + Kd, Kp + Ki * tau_f, Ki]
    den_C = [tau_f, 1.0, 0.0]

    # Produto C(s) * G(s)
    num_CG = np.convolve(num_C, num_G)
    den_CG = np.convolve(den_C, den_G)

    # Malha fechada para a saída: Tyr(s) = CG / (1 + CG) = num_CG / (den_CG + num_CG)
    den_cl = np.polyadd(den_CG, num_CG)

    # Verificação de estabilidade (critério de estabilidade via raízes do denominador)
    polos = np.roots(den_cl)
    if np.any(np.real(polos) >= 0):
        # Sistema instável
        t = np.arange(0, t_sim + dt, dt)
        return t, None, None, None, False

    # Função de transferência Tyr(s) = Y(s) / R(s)
    sys_yr = signal.TransferFunction(num_CG, den_cl)

    # Função de transferência Tur(s) = U(s) / R(s) = num_C * den_G / (den_CG + num_CG)
    num_ur = np.convolve(num_C, den_G)
    sys_ur = signal.TransferFunction(num_ur, den_cl)

    # Vetor de tempo
    t = np.arange(0, t_sim + dt, dt)
    u_degrau = np.ones_like(t)

    # Simulação linear da saída y(t) e do esforço u(t)
    _, y, _ = signal.lsim(sys_yr, U=u_degrau, T=t)
    _, u, _ = signal.lsim(sys_ur, U=u_degrau, T=t)
    e = u_degrau - y

    return t, y, e, u, True

### 3.3. Função Objetivo Multiobjetivo (ITAE + Regularização $L_2$)

Implementa-se a integração numérica aproximada pela regra dos trapézios para o índice ITAE ($J_1$) e para a energia do atuador ($J_2$):

$$
J(\boldsymbol{x}; \lambda) = \int_0^T t |e(t)| \, dt + \lambda \int_0^T u(t)^2 \, dt
$$

In [3]:
def funcao_custo(ganhos, lambda_reg=0.01, t_sim=5.0, dt=0.01):
    """
    Calcula a função custo multiobjetivo J = J1 + lambda * J2.
    Caso o sistema seja instável, retorna um valor penalizado muito alto.
    """
    t, y, e, u, estavel = simular_malha_fechada(ganhos, t_sim=t_sim, dt=dt)
    if not estavel:
        return 1e6  # Penalidade para instabilidade

    # Integral do Erro Absoluto Ponderado no Tempo (ITAE)
    J1 = np.trapz(t * np.abs(e), t)

    # Regularização L2 do esforço de controle
    J2 = np.trapz(u**2, t)

    # Custo ponderado total
    J_total = J1 + lambda_reg * J2

    return J_total

### 3.4. Resolução da Sintonia Ótima via Algoritmo Restrito (`SLSQP`)

Define-se o ponto de partida inicial $\boldsymbol{x}_0$ e os limites de caixa (*bounds*) operacionais:

$$
0.1 \le K_p \le 20.0, \quad 0.0 \le K_i \le 20.0, \quad 0.0 \le K_d \le 5.0
$$

Resolve-se o problema utilizando o método de Programação Quadrática Sequencial (`SLSQP`).

In [4]:
# Ponto de partida inicial (sintonia conservadora)
x0 = [2.0, 1.0, 0.2]

# Limites de caixa operacionais (Bounds)
bounds = [
    (0.1, 20.0),   # Kp
    (0.01, 20.0),  # Ki
    (0.0, 5.0)     # Kd
]

# Parâmetro nominal de regularização
lambda_nominal = 0.005

# Execução da otimização não linear
res = minimize(
    funcao_custo,
    x0,
    args=(lambda_nominal,),
    method='SLSQP',
    bounds=bounds,
    options={'disp': True, 'ftol': 1e-6, 'maxiter': 200}
)

x_otimo = res.x

print("\n" + "=" * 50)
print("RESULTADOS DA OTIMIZAÇÃO (SLSQP):")
print(f"Sucesso: {res.success} | Mensagem: {res.message}")
print(f"Ganhos Iniciais (x0) : Kp = {x0[0]:.3f}, Ki = {x0[1]:.3f}, Kd = {x0[2]:.3f}")
print(f"Ganhos Ótimos (x*)   : Kp = {x_otimo[0]:.3f}, Ki = {x_otimo[1]:.3f}, Kd = {x_otimo[2]:.3f}")
print(f"Valor do Custo Ótimo J : {res.fun:.4f}")
print("=" * 50)

AttributeError: module 'numpy' has no attribute 'trapz'

# 4 - Resultados e Análise Gráfica

# 5 - Discussão